# 02 - Eksperimen Baseline & Model Gabungan
Hasil dibuat oleh `src/run_experiments.py` (disimpan di `results/metrics.csv`). Jalankan dulu script itu, lalu **Run All** notebook ini.

**Model yang dibandingkan**
- Baseline: `majority`, `persistence` (naive), `arima`, `xgb:kurs` (XGBoost, fitur kurs saja)
- Model gabungan: XGBoost dengan fitur kurs + lexicon, kurs + TF-IDF, dan kurs + lexicon + TF-IDF

**Protokol**: model dilatih di train; data validasi dipakai memilih hyperparameter; data test dipakai satu kali untuk skor akhir.

In [ ]:
import sys; sys.path.append('../src')
import pandas as pd, matplotlib.pyplot as plt
from config import ROOT
from evaluate import cm_table
m = pd.read_csv(ROOT / 'results/metrics.csv')
p = pd.read_csv(ROOT / 'results/predictions_test.csv')
cols = ['model', 'DA', 'F1_macro', 'share_pred1']
print('VALIDASI'); print(m[m.split == 'val'][cols].round(3).to_string(index=False))
print('\nTEST');     print(m[m.split == 'test'][cols].round(3).to_string(index=False))

`share_pred1` = proporsi hari yang diprediksi "rupiah melemah". Kalau nilainya mendekati 1, model hampir selalu menebak kelas mayoritas. Dalam kasus itu DA-nya mirip majority baseline dan F1 macro-nya rendah, jadi DA harus dibaca bersama F1 macro.

In [ ]:
# Perbandingan Directional Accuracy di data test
t = m[m.split == 'test'].set_index('model')['DA']
maj = t['majority']
ax = t.plot.barh(figsize=(8, 4))
ax.axvline(maj, color='r', ls='--', label=f'majority ({maj:.3f})')
ax.set_xlim(0.4, 0.65); ax.set_xlabel('Directional accuracy (test)'); ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
# Confusion matrix di data test
for name in ['majority', 'xgb:kurs', 'xgb:kurs+lexicon', 'xgb:kurs+tfidf', 'xgb:kurs+lexicon+tfidf']:
    print(name); print(cm_table(p['y'], p[name]).to_string(), '\n')

## Cara membaca hasil
- Bandingkan tiap model gabungan dengan `xgb:kurs`: kalau DA dan F1 macro-nya tidak naik, fitur NLP belum membantu.
- Bandingkan semua model dengan `majority` dan `persistence`. Dengan 241 hari test, satu standar error DA sekitar 3 poin, jadi selisih kecil bisa saja kebetulan.
- Tulis ringkasan temuan tim di sel ini setelah melihat angkanya.